Compare original-launch and revival complexion CMYK by shade depth group.
*Co-authored with CoCo*

In [ ]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

for weight_file in ['ClashGrotesk-Extralight.ttf', 'ClashGrotesk-Light.ttf',
                    'ClashGrotesk-Regular.ttf', 'ClashGrotesk-Medium.ttf',
                    'ClashGrotesk-Semibold.ttf', 'ClashGrotesk-Bold.ttf']:
    fm.fontManager.addfont(f'fonts/Clash_Grotesk/{weight_file}')

plt.rcParams['font.family'] = 'Clash Grotesk'
plt.rcParams['font.weight'] = 'light'

In [ ]:
%%sql -r dataframe_1
USE DATABASE MARCJACOBS_BEAUTY_REVIVAL;

In [ ]:
%%sql -r dataframe_2
WITH complexion_shades AS (
  SELECT
    era,
    product_type,
    product_name,
    shade_depth_group,
    undertone_inferred_cmyk,
    undertone_confidence,
    c_pct,
    m_pct,
    y_pct,
    k_pct
  FROM MARCJACOBS_BEAUTY_REVIVAL.ANALYTICS.SHADES
  WHERE product_type IN ('Foundation', 'Concealer', 'Bronzer')
    AND c_pct IS NOT NULL
    AND m_pct IS NOT NULL
    AND y_pct IS NOT NULL
    AND k_pct IS NOT NULL
    AND shade_depth_group IS NOT NULL
), depth_summary AS (
  SELECT
    era,
    product_type,
    shade_depth_group,
    COUNT(*) AS shade_rows,
    ROUND(AVG(c_pct), 2) AS avg_c_pct,
    ROUND(AVG(m_pct), 2) AS avg_m_pct,
    ROUND(AVG(y_pct), 2) AS avg_y_pct,
    ROUND(AVG(k_pct), 2) AS avg_k_pct,
    ROUND(AVG(c_pct + m_pct + y_pct + k_pct), 2) AS avg_total_ink
  FROM complexion_shades
  GROUP BY era, product_type, shade_depth_group
), pivoted AS (
  SELECT
    product_type,
    shade_depth_group,
    MAX(CASE WHEN era = 'original_launch' THEN shade_rows END) AS original_shade_rows,
    MAX(CASE WHEN era = 'revival' THEN shade_rows END) AS revival_shade_rows,
    MAX(CASE WHEN era = 'original_launch' THEN avg_c_pct END) AS original_avg_c_pct,
    MAX(CASE WHEN era = 'revival' THEN avg_c_pct END) AS revival_avg_c_pct,
    MAX(CASE WHEN era = 'original_launch' THEN avg_m_pct END) AS original_avg_m_pct,
    MAX(CASE WHEN era = 'revival' THEN avg_m_pct END) AS revival_avg_m_pct,
    MAX(CASE WHEN era = 'original_launch' THEN avg_y_pct END) AS original_avg_y_pct,
    MAX(CASE WHEN era = 'revival' THEN avg_y_pct END) AS revival_avg_y_pct,
    MAX(CASE WHEN era = 'original_launch' THEN avg_k_pct END) AS original_avg_k_pct,
    MAX(CASE WHEN era = 'revival' THEN avg_k_pct END) AS revival_avg_k_pct,
    MAX(CASE WHEN era = 'original_launch' THEN avg_total_ink END) AS original_avg_total_ink,
    MAX(CASE WHEN era = 'revival' THEN avg_total_ink END) AS revival_avg_total_ink
  FROM depth_summary
  GROUP BY product_type, shade_depth_group
)
SELECT
  product_type,
  shade_depth_group,
  original_shade_rows,
  revival_shade_rows,
  original_avg_c_pct,
  revival_avg_c_pct,
  revival_avg_c_pct - original_avg_c_pct AS delta_c_pct,
  original_avg_m_pct,
  revival_avg_m_pct,
  revival_avg_m_pct - original_avg_m_pct AS delta_m_pct,
  original_avg_y_pct,
  revival_avg_y_pct,
  revival_avg_y_pct - original_avg_y_pct AS delta_y_pct,
  original_avg_k_pct,
  revival_avg_k_pct,
  revival_avg_k_pct - original_avg_k_pct AS delta_k_pct,
  original_avg_total_ink,
  revival_avg_total_ink,
  revival_avg_total_ink - original_avg_total_ink AS delta_total_ink
FROM pivoted
WHERE original_shade_rows IS NOT NULL
   OR revival_shade_rows IS NOT NULL
ORDER BY product_type,
  CASE shade_depth_group
    WHEN 'very_fair' THEN 1
    WHEN 'fair' THEN 2
    WHEN 'light' THEN 3
    WHEN 'light_medium' THEN 4
    WHEN 'medium' THEN 5
    WHEN 'medium_deep' THEN 6
    WHEN 'deep' THEN 7
    WHEN 'very_deep' THEN 8
    WHEN 'universal' THEN 9
    WHEN 'brightener' THEN 10
    WHEN 'corrector' THEN 11
    ELSE 12
  END;

In [ ]:
%%sql -r complexion_scatter_data
WITH complexion_scatter AS (
  SELECT
    era,
    product_type,
    product_name,
    shade_name,
    shade_depth_group,
    c_pct,
    m_pct,
    y_pct,
    k_pct,
    ROUND(k_pct + 0.35 * m_pct + 0.15 * y_pct, 2) AS depth_score,
    ROUND(y_pct - m_pct, 2) AS undertone_score,
    CASE
      WHEN y_pct - m_pct >= 12 THEN 'yellow'
      WHEN y_pct - m_pct >= 4 THEN 'warm'
      WHEN y_pct - m_pct <= -8 THEN 'pink'
      WHEN c_pct - y_pct >= 8 THEN 'olive'
      ELSE 'neutral'
    END AS undertone_label
  FROM MARCJACOBS_BEAUTY_REVIVAL.ANALYTICS.SHADES
  WHERE product_type IN ('Foundation', 'Concealer', 'Bronzer')
    AND c_pct IS NOT NULL
    AND m_pct IS NOT NULL
    AND y_pct IS NOT NULL
    AND k_pct IS NOT NULL
)
SELECT
  era,
  product_type,
  product_name,
  shade_name,
  shade_depth_group,
  c_pct,
  m_pct,
  y_pct,
  k_pct,
  depth_score,
  undertone_score,
  undertone_label
FROM complexion_scatter
ORDER BY era, product_type, depth_score, undertone_score;

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = complexion_scatter_data.to_pandas() if hasattr(complexion_scatter_data, 'to_pandas') else complexion_scatter_data
df = df.copy()

era_order = ['original_launch', 'revival']
era_labels = {'original_launch': '2013 Collection', 'revival': '2026 Revival'}
era_annotations = {
    'original_launch': 'Foundation, concealer, and bronzer across all shade depths',
    'revival': 'Only bronzer remains in the revival complexion lineup',
}
plot_df = df[df['ERA'].isin(era_order)].copy()
plot_df.columns = [column.lower() for column in plot_df.columns]

def cmyk_to_rgb(c, m, y, k):
    c, m, y, k = c / 100, m / 100, y / 100, k / 100
    r = (1 - c) * (1 - k)
    g = (1 - m) * (1 - k)
    b = (1 - y) * (1 - k)
    return (r, g, b)

plot_df['rgb'] = plot_df.apply(
    lambda row: cmyk_to_rgb(row['c_pct'], row['m_pct'], row['y_pct'], row['k_pct']), axis=1
)

shape_map = {
    'Foundation': 'o',
    'Concealer': 's',
    'Bronzer': 'D',
}

def get_extreme_labels(era_df, n=5):
    extremes = set()
    if len(era_df) <= n * 2:
        return set(era_df['shade_name'])
    extremes.update(era_df.nsmallest(n, 'depth_score')['shade_name'])
    extremes.update(era_df.nlargest(n, 'depth_score')['shade_name'])
    extremes.update(era_df.nsmallest(2, 'undertone_score')['shade_name'])
    extremes.update(era_df.nlargest(2, 'undertone_score')['shade_name'])
    return extremes

def plot_complexion(layout='horizontal'):
    if layout == 'horizontal':
        fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True, sharey=True)
        order = era_order
    else:
        fig, axes = plt.subplots(2, 1, figsize=(9, 14), sharex=True)
        order = era_order

    for i, (axis, era) in enumerate(zip(axes.flat, order)):
        era_df = plot_df[plot_df['era'] == era].copy()
        axis.axhline(0, color='#d9d9d9', linewidth=1, zorder=0)

        for product_type, type_df in era_df.groupby('product_type'):
            colors = list(type_df['rgb'])
            marker = shape_map.get(product_type, 'o')
            axis.scatter(
                type_df['depth_score'],
                type_df['undertone_score'],
                s=90,
                alpha=0.85,
                c=colors,
                edgecolors='black',
                linewidths=0.5,
                marker=marker,
                label=product_type,
            )

        axis.set_title(era_labels[era], fontsize=13, fontweight='medium', pad=20)
        axis.text(0.5, 1.01, era_annotations[era],
                  transform=axis.transAxes, ha='center', fontsize=11,
                  fontweight='normal', color='#444')
        axis.set_xlabel('Shade depth, from lighter to deeper', fontsize=11, fontweight='normal')
        axis.grid(alpha=0.2)
        axis.tick_params(labelsize=9)

        if layout == 'vertical' or i == 0:
            axis.set_ylabel('Undertone, from pink to yellow', fontsize=11, fontweight='normal')

    handles, labels = axes.flat[0].get_legend_handles_labels()
    if layout == 'horizontal':
        axes.flat[0].legend(handles, labels, title='Product type',
                            loc='lower left', bbox_to_anchor=(0, 1.15),
                            fontsize=10, title_fontsize=10,
                            frameon=False, borderaxespad=0)
        fig.suptitle('COMPLEXION SHADES BY DEPTH AND UNDERTONE',
                     fontsize=18, fontweight='medium', y=0.95)
        plt.tight_layout()
    else:
        axes.flat[0].legend(handles, labels, title='Product type',
                            loc='lower left', bbox_to_anchor=(0, 1.08),
                            fontsize=10, title_fontsize=10,
                            frameon=False, borderaxespad=0)
        plt.tight_layout()
        plt.subplots_adjust(hspace=0.22, top=0.94)
        fig.suptitle('COMPLEXION SHADES BY DEPTH AND UNDERTONE',
                     fontsize=18, fontweight='medium', y=1.05)
    plt.show()

plot_complexion('horizontal')
plot_complexion('vertical')

In [ ]:
# Foundation & Concealer scatter plot
fc_df = plot_df[plot_df['product_type'].isin(['Foundation', 'Concealer'])].copy()

shape_map_fc = {
    'Foundation': 'o',
    'Concealer': 's',
}

fc_annotations = {
    'original_launch': 'Shades cluster in the light-to-medium, pink-undertone range',
    'revival': 'No foundation or concealer in the revival lineup',
}

def plot_fc(layout='horizontal'):
    if layout == 'horizontal':
        fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True, sharey=True)
        order = era_order
    else:
        fig, axes = plt.subplots(2, 1, figsize=(9, 14), sharex=True)
        order = era_order

    for i, (axis, era) in enumerate(zip(axes.flat, order)):
        era_df = fc_df[fc_df['era'] == era].copy()
        axis.axhline(0, color='#d9d9d9', linewidth=1, zorder=0)

        if not era_df.empty:
            for product_type, type_df in era_df.groupby('product_type'):
                colors = list(type_df['rgb'])
                marker = shape_map_fc.get(product_type, 'o')
                axis.scatter(
                    type_df['depth_score'],
                    type_df['undertone_score'],
                    s=100,
                    alpha=0.85,
                    c=colors,
                    edgecolors='black',
                    linewidths=0.5,
                    marker=marker,
                    label=product_type,
                )
            extremes = get_extreme_labels(era_df)
            for _, row in era_df.iterrows():
                if row['shade_name'] in extremes:
                    axis.annotate(
                        row['shade_name'],
                        (row['depth_score'], row['undertone_score']),
                        textcoords='offset points',
                        xytext=(5, 4),
                        fontsize=9,
                        fontweight='normal',
                        alpha=0.85,
                    )
        else:
            axis.text(0.5, 0.5, 'No foundation or concealer in revival',
                      transform=axis.transAxes, ha='center', va='center',
                      fontsize=13, color='#999', style='italic', fontweight='normal')

        axis.set_xlim(0, 100)
        axis.set_title(era_labels[era], fontsize=13, fontweight='medium', pad=20)
        axis.text(0.5, 1.01, fc_annotations[era],
                  transform=axis.transAxes, ha='center', fontsize=11,
                  fontweight='normal', color='#444')
        axis.set_xlabel('Shade depth, from lighter to deeper', fontsize=11, fontweight='normal')
        axis.grid(alpha=0.2)
        axis.tick_params(labelsize=9)

        if layout == 'vertical' or i == 0:
            axis.set_ylabel('Undertone, from pink to yellow', fontsize=11, fontweight='normal')

    handles, labels = axes.flat[0].get_legend_handles_labels()
    if layout == 'horizontal':
        if handles:
            axes.flat[0].legend(handles, labels, title='Product type',
                                loc='lower left', bbox_to_anchor=(0, 1.15),
                                fontsize=10, title_fontsize=10,
                                frameon=False, borderaxespad=0)
        fig.suptitle('FOUNDATION & CONCEALER: NOT IN THE REVIVAL',
                     fontsize=18, fontweight='medium', y=0.95)
        plt.tight_layout()
    else:
        if handles:
            axes.flat[0].legend(handles, labels, title='Product type',
                                loc='lower left', bbox_to_anchor=(0, 1.08),
                                fontsize=10, title_fontsize=10,
                                frameon=False, borderaxespad=0)
        plt.tight_layout()
        plt.subplots_adjust(hspace=0.22, top=0.94)
        fig.suptitle('FOUNDATION & CONCEALER: NOT IN THE REVIVAL',
                     fontsize=18, fontweight='medium', y=1.05)
    plt.show()

plot_fc('horizontal')
plot_fc('vertical')

In [ ]:
# Bronzer scatter plot
bronzer_df = plot_df[plot_df['product_type'] == 'Bronzer'].copy()

bronzer_annotations = {
    'original_launch': 'Single bronzer product with limited shade range',
    'revival': 'Expanded to 9 shades across fair to deep',
}

def plot_bronzer(layout='horizontal'):
    if layout == 'horizontal':
        fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True, sharey=True)
        order = era_order
    else:
        fig, axes = plt.subplots(2, 1, figsize=(9, 14), sharex=True)
        order = era_order

    for i, (axis, era) in enumerate(zip(axes.flat, order)):
        era_df = bronzer_df[bronzer_df['era'] == era].copy()
        axis.axhline(0, color='#d9d9d9', linewidth=1, zorder=0)

        if not era_df.empty:
            colors = list(era_df['rgb'])
            axis.scatter(
                era_df['depth_score'],
                era_df['undertone_score'],
                s=100,
                alpha=0.85,
                c=colors,
                edgecolors='black',
                linewidths=0.5,
                marker='D',
            )
            extremes = get_extreme_labels(era_df)
            for _, row in era_df.iterrows():
                if row['shade_name'] in extremes:
                    axis.annotate(
                        row['shade_name'],
                        (row['depth_score'], row['undertone_score']),
                        textcoords='offset points',
                        xytext=(5, 4),
                        fontsize=9,
                        fontweight='normal',
                        alpha=0.85,
                    )
        else:
            axis.text(0.5, 0.5, 'No bronzer shades',
                      transform=axis.transAxes, ha='center', va='center',
                      fontsize=13, color='#999', style='italic', fontweight='normal')

        axis.set_xlim(0, 100)
        axis.set_title(era_labels[era], fontsize=13, fontweight='medium', pad=20)
        axis.text(0.5, 1.01, bronzer_annotations[era],
                  transform=axis.transAxes, ha='center', fontsize=11,
                  fontweight='normal', color='#444')
        axis.set_xlabel('Shade depth, from lighter to deeper', fontsize=11, fontweight='normal')
        axis.grid(alpha=0.2)
        axis.tick_params(labelsize=9)

        if layout == 'vertical' or i == 0:
            axis.set_ylabel('Undertone, from pink to yellow', fontsize=11, fontweight='normal')

    if layout == 'horizontal':
        fig.suptitle('BRONZER SHADE RANGE EXPANDED IN THE REVIVAL',
                     fontsize=18, fontweight='medium', y=0.95)
        plt.tight_layout()
    else:
        plt.tight_layout()
        plt.subplots_adjust(hspace=0.22, top=0.94)
        fig.suptitle('BRONZER SHADE RANGE EXPANDED IN THE REVIVAL',
                     fontsize=18, fontweight='medium', y=1.05)
    plt.show()

plot_bronzer('horizontal')
plot_bronzer('vertical')